SOAL 1 — Data Loading & Profiling Awal

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StructType, StructField, StringType, DateType, TimestampType

In [2]:
spark = SparkSession.builder \
    .appName("ecommerce_profiling") \
    .master("local[*]") \
    .getOrCreate()

BASE = "dataset/"


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/giwang.kintan/Documents/DE/pyspark_assignment/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/17 22:59:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/17 22:59:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
spark.sparkContext.setLogLevel("ERROR")

In [4]:
orders_schema = StructType([
    StructField("order_id",     StringType()),
    StructField("user_id",      StringType()),
    StructField("order_date",   DateType()),
    StructField("total_amount", DoubleType()),
])
order_items_schema = StructType([
    StructField("order_id",      StringType()),
    StructField("product_id",    StringType()),
    StructField("quantity",      IntegerType()),
    StructField("price",         DoubleType()),
])
products_schema = StructType([
    StructField("product_id",   StringType()),
    StructField("product_name", StringType()),
    StructField("category",     StringType()),
    StructField("price",        DoubleType()),
])
users_schema = StructType([
    StructField("user_id",     StringType()),
    StructField("name",        StringType()),
    StructField("city",        StringType()),
    StructField("signup_date", DateType()),
])

In [5]:
def load(name, schema):
    return (spark.read.option("header", True)
            .option("dateFormat", "yyyy-MM-dd")
            .option("mode", "PERMISSIVE")
            .schema(schema)
            .csv(f"{BASE}{name}.csv"))

In [6]:

orders      = load("orders", orders_schema)
order_items = load("order_items", order_items_schema)
products    = load("products", products_schema)
users       = load("users", users_schema)

dfs = {"orders": orders, "order_items": order_items,
       "products": products, "users": users}

In [7]:
print("=== Jumlah baris tiap DataFrame ===")
for table, row in dfs.items():
    print(f"{table}: {row.count()}")


print("\n=== cek null per kolom ===")
for table, row in dfs.items():
    print(f"\n[{table}]")
    row.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                for c in row.columns]).show(truncate=False)

print("\n=== Sample 5 baris ===")
for table, row in dfs.items():
    print(f"\n[{table}]")
    row.show(5, truncate=False)


=== Jumlah baris tiap DataFrame ===
orders: 18
order_items: 20
products: 10
users: 14

=== cek null per kolom ===

[orders]
+--------+-------+----------+------------+
|order_id|user_id|order_date|total_amount|
+--------+-------+----------+------------+
|1       |1      |0         |0           |
+--------+-------+----------+------------+


[order_items]
+--------+----------+--------+-----+
|order_id|product_id|quantity|price|
+--------+----------+--------+-----+
|1       |1         |0       |0    |
+--------+----------+--------+-----+


[products]
+----------+------------+--------+-----+
|product_id|product_name|category|price|
+----------+------------+--------+-----+
|1         |0           |0       |0    |
+----------+------------+--------+-----+


[users]
+-------+----+----+-----------+
|user_id|name|city|signup_date|
+-------+----+----+-----------+
|1      |0   |0   |0          |
+-------+----+----+-----------+


=== Sample 5 baris ===

[orders]
+--------+-------+----------+--------

In [8]:
print("=== jumlah orders dengan total_amount <= 0 ===")

orders_bad_amount = orders.filter(F.col("total_amount") <= 0).count()
print(f"orders: total_amount <= 0: {orders_bad_amount}")

print("\n=== jumlah order_items dengan quantity <= 0 atau price <= 0 ===")
items_bad_qty_or_price = order_items.filter(
    (F.col("quantity") <= 0) | (F.col("price") <= 0)
).count()

print(f"order_items: quantity <= 0 or price <= 0: {items_bad_qty_or_price}")

print("\n=== jumlah duplikasi key utama (order_id di orders, product_id di products, user_id di users) ===")
def count_duplicates(df, key):
    return df.groupBy(key).count().filter(F.col("count") > 1).count()

print(f"orders: duplicate order_id: {count_duplicates(orders, 'order_id')}")
print(f"products: duplicate product_id: {count_duplicates(products, 'product_id')}")
print(f"users: duplicate user_id: {count_duplicates(users, 'user_id')}")




=== jumlah orders dengan total_amount <= 0 ===
orders: total_amount <= 0: 2

=== jumlah order_items dengan quantity <= 0 atau price <= 0 ===
order_items: quantity <= 0 or price <= 0: 2

=== jumlah duplikasi key utama (order_id di orders, product_id di products, user_id di users) ===
orders: duplicate order_id: 1
products: duplicate product_id: 1
users: duplicate user_id: 1


SOAL 2 — Cleaning & Standardization
Perintah 1:
Bersihkan data:
drop baris orders yang order_id atau user_id null,
drop baris order_items yang order_id atau product_id null,
drop baris products yang product_id null,
drop baris users yang user_id null.

Perintah 2:
Standardisasi kolom teks:
city di users → lowercase + trim,
category di products → lowercase + trim.
Perintah 3:
Deduplication berbasis key:
orders dedupe by order_id,
order_items dedupe by kombinasi (order_id,product_id),
products dedupe by product_id,
users dedupe by user_id.
 Tunjukkan jumlah baris sebelum vs sesudah dedupe.


In [9]:
# ================ Perintah 1: Bersihkan data ======================

print("=== drop baris yang null===")
orders_cleaned = orders.dropna(subset=["order_id", "user_id"])
order_items_cleaned = order_items.dropna(subset=[ "order_id", "product_id"])
products_cleaned = products.dropna(subset=["product_id"])
users_cleaned = users.dropna(subset=["user_id"])

print("drop baris yang null selesai")


=== drop baris yang null===
drop baris yang null selesai


In [10]:

# ================ Perintah 2: Standardisasi kolom teks ======================
users_cleaned = users_cleaned.withColumn("city", F.lower(F.trim(F.col("city"))))
products_cleaned = products_cleaned.withColumn("category", F.lower(F.trim(F.col("category"))))

users_cleaned.show(5, truncate=False)
products_cleaned.show(5, truncate=False)


+-------+-------------+--------+-----------+
|user_id|name         |city    |signup_date|
+-------+-------------+--------+-----------+
|U001   |Andi Pratama |jakarta |2023-01-15 |
|U002   |Budi Santoso |surabaya|2023-03-20 |
|U002   |Budi Santoso |surabaya|2023-03-20 |
|U003   |Citra Dewi   |bandung |2023-06-10 |
|U004   |Dimas Anggara|jakarta |2023-07-01 |
+-------+-------------+--------+-----------+
only showing top 5 rows
+----------+-------------+----------+---------+
|product_id|product_name |category  |price    |
+----------+-------------+----------+---------+
|P001      |Smartphone X |elektronik|3500000.0|
|P001      |Smartphone X |elektronik|3500000.0|
|P002      |Laptop Pro   |elektronik|8500000.0|
|P003      |Kemeja Casual|fashion   |150000.0 |
|P004      |Celana Jeans |fashion   |250000.0 |
+----------+-------------+----------+---------+
only showing top 5 rows


In [11]:
# ================ Perintah 3: Deduplication berbasis key: ======================
orders_dedup = orders_cleaned.dropDuplicates(["order_id"])
order_items_dedup = order_items_cleaned.dropDuplicates(["order_id", "product_id"])
products_dedup = products_cleaned.dropDuplicates(["product_id"])
users_dedup = users_cleaned.dropDuplicates(["user_id"])

# ================ Perintah 4: Data sebelum vs sesudah dedupe ======================
pairs = [
    ("orders",      orders_cleaned,      orders_dedup),
    ("order_items", order_items_cleaned, order_items_dedup),
    ("products",    products_cleaned,    products_dedup),
    ("users",       users_cleaned,       users_dedup),
]
rows = []
for name, before_df, after_df in pairs:
    before, after = before_df.count(), after_df.count()
    rows.append((name, before, after, before - after))

print("\n=== jumlah baris sebelum vs sesudah deduplication ===")
dedup_summary = spark.createDataFrame(
    rows, ["table", "before_dedupe", "after_dedupe", "rows_removed"]
)
dedup_summary.show()



=== jumlah baris sebelum vs sesudah deduplication ===


+-----------+-------------+------------+------------+
|      table|before_dedupe|after_dedupe|rows_removed|
+-----------+-------------+------------+------------+
|     orders|           16|          15|           1|
|order_items|           18|          17|           1|
|   products|            9|           8|           1|
|      users|           13|          12|           1|
+-----------+-------------+------------+------------+



SOAL 3 — Data Enrichment & Feature Engineering
Perintah 1:
Di order_items, buat kolom turunan:
total_item_value = quantity * price.
Perintah 2:
Di orders, buat kolom kategorisasi menggunakan when/otherwise:
order_value_flag dengan aturan:
high_value jika total_amount > 500000, selain itu normal.
Perintah 3:
Buat kolom segmentasi user di users:
user_tenure_days = datediff(order_date_max_per_user, signup_date) (Hint: butuh join/agg kecil untuk order_date_max_per_user).
tenure_segment = new (<=30 hari), mature (31–180), loyal (>180).

In [13]:
# ================ Perintah 1: Di order_items, buat kolom turunan ======================
order_items_dedup = order_items_dedup.withColumn("total_price", F.col("quantity") * F.col("price"))
print("\n=== order_items dengan kolom turunan total_price ===")
order_items_dedup.show(5, truncate=False)
# ================ Perintah 2: Di orders, buat kolom kategorisasi menggunakan when/otherwise:======================
orders_dedup = orders_dedup.withColumn("order_value_flag", F.when(F.col("total_amount") > 500000, "high_value").otherwise("normal"))   
print("\n=== orders dengan kolom kategorisasi order_value_flag ===")
orders_dedup.show(5, truncate=False) 



=== order_items dengan kolom turunan total_price ===
+--------+----------+--------+---------+-----------+
|order_id|product_id|quantity|price    |total_price|
+--------+----------+--------+---------+-----------+
|O015    |P007      |2       |450000.0 |900000.0   |
|O011    |P001      |1       |3500000.0|3500000.0  |
|O007    |P003      |-1      |150000.0 |-150000.0  |
|O009    |P007      |1       |450000.0 |450000.0   |
|O008    |P004      |1       |-50000.0 |-50000.0   |
+--------+----------+--------+---------+-----------+
only showing top 5 rows

=== orders dengan kolom kategorisasi order_value_flag ===
+--------+-------+----------+------------+----------------+
|order_id|user_id|order_date|total_amount|order_value_flag|
+--------+-------+----------+------------+----------------+
|O001    |U001   |2023-10-01|3500000.0   |high_value      |
|O002    |U001   |2023-10-15|400000.0    |normal          |
|O003    |U002   |2023-10-02|8500000.0   |high_value      |
|O004    |U003   |2023-10-

In [19]:
# ================ Perintah 3: Di users, buat kolom segmentasi ======================
last_order = (
    orders_dedup
    .groupBy("user_id")
    .agg(F.max("order_date").alias("order_date_max_per_user"))
)

# 3b. Join ke users (left join agar user tanpa order tetap ada)
user_with_last_order = users_dedup.join(last_order, on="user_id", how="left")

# 3c. Hitung tenure (hari)
user_with_last_order = user_with_last_order.withColumn(
    "user_tenure_days",
    F.datediff(F.col("order_date_max_per_user"), F.col("signup_date"))
)

# 3d. Segmentasi
user_with_last_order = user_with_last_order.withColumn(
    "tenure_segment",
    F.when(F.col("user_tenure_days").isNull(), "no_orders")
     .when(F.col("user_tenure_days") <= 30, "new")
     .when(F.col("user_tenure_days") <= 180, "mature")
     .otherwise("loyal")
)


print("\n=== users dengan kolom turunan user_tenure_days dan tenure_segment ===")
user_with_last_order.show(5, truncate=False)
user_with_last_order.groupBy("tenure_segment").count().show()


=== users dengan kolom turunan user_tenure_days dan tenure_segment ===
+-------+-------------+--------+-----------+-----------------------+----------------+--------------+
|user_id|name         |city    |signup_date|order_date_max_per_user|user_tenure_days|tenure_segment|
+-------+-------------+--------+-----------+-----------------------+----------------+--------------+
|U001   |Andi Pratama |jakarta |2023-01-15 |2023-10-20             |278             |loyal         |
|U002   |Budi Santoso |surabaya|2023-03-20 |2023-10-21             |215             |loyal         |
|U003   |Citra Dewi   |bandung |2023-06-10 |2023-10-03             |115             |mature        |
|U004   |Dimas Anggara|jakarta |2023-07-01 |2023-10-04             |95              |mature        |
|U005   |Eka Putri    |medan   |2023-08-15 |2023-10-05             |51              |mature        |
+-------+-------------+--------+-----------+-----------------------+----------------+--------------+
only showing top 5 

SOAL 4 — Join & Aggregation (Business Metrics)
Perintah 1:
Bangun DataFrame “fact” hasil join:
orders ⟷ order_items ⟷ products ⟷ users
dengan key:
orders ↔ order_items: order_id
order_items ↔ products: product_id
orders ↔ users: user_id
Perintah 2:
Hitung metrik per category per day:
daily_gmv_category = sum(total_item_value)
daily_orders_category = countDistinct(order_id)
daily_items_sold = sum(quantity)
 Urutkan per hari dan GMV terbesar.
Perintah 3:
Hitung metrik per city:
total GMV kota,
AOV kota (avg total_amount),
jumlah customer unik kota,
top 3 kategori kontribusi terbesar per kota (boleh pakai window rank).



In [20]:
#================= Perintah 1: Bangun DataFrame “fact” hasil join ======================
fact_df = (
    orders_dedup
    .join(order_items_dedup, on="order_id", how="inner")
    .join(products_dedup, on="product_id", how="inner")
    .join(users_dedup, on="user_id", how="inner")
)


print("\n=== fact_df hasil join ===")
fact_df.show(5, truncate=False)


=== fact_df hasil join ===
+-------+----------+--------+----------+------------+----------------+--------+---------+-----------+-----------------+------------+---------+-------------+--------+-----------+
|user_id|product_id|order_id|order_date|total_amount|order_value_flag|quantity|price    |total_price|product_name     |category    |price    |name         |city    |signup_date|
+-------+----------+--------+----------+------------+----------------+--------+---------+-----------+-----------------+------------+---------+-------------+--------+-----------+
|U001   |P001      |O001    |2023-10-01|3500000.0   |high_value      |1       |3500000.0|3500000.0  |Smartphone X     |elektronik  |3500000.0|Andi Pratama |jakarta |2023-01-15 |
|U001   |P008      |O002    |2023-10-15|400000.0    |normal          |2       |200000.0 |400000.0   |Headset Bluetooth|elektronik  |200000.0 |Andi Pratama |jakarta |2023-01-15 |
|U002   |P002      |O003    |2023-10-02|8500000.0   |high_value      |1       |850

In [21]:
#================= Perintah 2: Hitung metrik per category per day: ======================
print("metrik per category per day:")
metrics_per_category_per_day = (
    fact_df
    .groupBy("order_date", "category")
    .agg(
        F.sum("total_amount").alias("daily_gmv_category"),
        F.countDistinct("order_id").alias("daily_orders_category"),
        F.countDistinct("user_id").alias("daily_customers_category")
    )
    .orderBy("order_date", "daily_gmv_category")
)
metrics_per_category_per_day.show(20, truncate=False)

metrik per category per day:
+----------+------------+------------------+---------------------+------------------------+
|order_date|category    |daily_gmv_category|daily_orders_category|daily_customers_category|
+----------+------------+------------------+---------------------+------------------------+
|2023-10-01|elektronik  |3500000.0         |1                    |1                       |
|2023-10-02|elektronik  |8500000.0         |1                    |1                       |
|2023-10-03|fashion     |150000.0          |1                    |1                       |
|2023-10-04|rumah tangga|300000.0          |1                    |1                       |
|2023-10-05|rumah tangga|1200000.0         |1                    |1                       |
|2023-10-06|fashion     |-50000.0          |1                    |1                       |
|2023-10-07|fashion     |0.0               |1                    |1                       |
|2023-10-08|fashion     |1400000.0         |1      

In [27]:
from pyspark.sql.window import Window
#================= Perintah 2: Hitung metrik per city ======================
print("metrik per city:")

# Hitung metrik per city:
# total GMV kota,
# AOV kota (avg total_amount),
# jumlah customer unik kota,
# top 3 kategori kontribusi terbesar per kota (boleh pakai window rank).


metrics_per_city = (
    fact_df
    .groupBy("city")
    .agg(
        F.sum("total_amount").alias("total_gmv_city"),
        F.avg("total_amount").alias("aov_city"),
        F.countDistinct("user_id").alias("unique_customers_city")
    )
    .orderBy("total_gmv_city", ascending=False)
)

metrics_per_city.show(20, truncate=False)

print("\n=== top 3 kategori kontribusi terbesar per kota ===")

gmv_category_per_city = (
    fact_df
    .groupBy("city", "category")
    .agg(F.sum("total_amount").alias("total_gmv_category"))
)

window_spec = Window.partitionBy("city").orderBy(F.desc("total_gmv_category"))
top_categories_per_city = (
    gmv_category_per_city
    .withColumn("rank", F.rank().over(window_spec))
    .filter(F.col("rank") <= 3)
    .select("city", "category", "total_gmv_category", "rank")
    .orderBy("city", "rank")
)
top_categories_per_city.show(20, truncate=False)


metrik per city:
+----------+--------------+---------+---------------------+
|city      |total_gmv_city|aov_city |unique_customers_city|
+----------+--------------+---------+---------------------+
|surabaya  |1.015E7       |2537500.0|2                    |
|bandung   |7500000.0     |1875000.0|3                    |
|jakarta   |4850000.0     |970000.0 |3                    |
|medan     |1200000.0     |1200000.0|1                    |
|makassar  |900000.0      |900000.0 |1                    |
|yogyakarta|600000.0      |600000.0 |1                    |
|semarang  |0.0           |0.0      |1                    |
+----------+--------------+---------+---------------------+


=== top 3 kategori kontribusi terbesar per kota ===
+----------+------------+------------------+----+
|city      |category    |total_gmv_category|rank|
+----------+------------+------------------+----+
|bandung   |elektronik  |7400000.0         |1   |
|bandung   |fashion     |100000.0          |2   |
|jakarta   |elektro

SOAL 5 — Window Analytics & Insight Final
Perintah 1:
Gunakan window function per user untuk membuat kolom:
user_total_spending (sum total_amount per user),
user_total_orders (count order per user),
user_avg_order_value (avg total_amount per user),
user_rank global berdasarkan user_total_spending (dense_rank).
Perintah 2:
Buat analisis cumulative GMV harian:
gmv_daily,
cumulative_gmv (sum gmv_daily over order by date).


Perintah 3:
Ambil Top 10 customers (rank 1–10) dan tampilkan:
user_id, name, city, user_total_orders, user_total_spending, user_avg_order_value.
Tambahkan interpretasi singkat 4–6 kalimat: pola apa yang kamu lihat? (misal kota dominan, AOV tinggi tapi order sedikit, dll).


In [ ]:
#================= Perintah 1: Gunakan window function per user untuk membuat kolom======================
user_window = Window.partitionBy("user_id")

orders_with_user_metrics = (
    orders_dedup 
    .withColumn("user_total_spending", F.sum("total_amount").over(user_window))
    .withColumn("user_total_orders", F.count("order_id").over(user_window))
    .withColumn("user_avg_order_value", F.avg("total_amount").over(user_window))
)

user_window = Window.orderBy(F.desc("user_total_spending"))

user_summary = (
    orders_with_user_metrics
    .select("user_id", "user_total_spending", "user_total_orders", "user_avg_order_value")
    .dropDuplicates(["user_id"])
    .withColumn("user_rank", F.dense_rank().over(user_window))
)

print("\n=== user_summary dengan kolom turunan user_total_spending, user_total_orders, user_avg_order_value, user_rank ===")
user_summary.orderBy("user_rank").show(20, truncate=False)



=== user_summary dengan kolom turunan user_total_spending, user_total_orders, user_avg_order_value, user_rank ===
+-------+-------------------+-----------------+--------------------+---------+
|user_id|user_total_spending|user_total_orders|user_avg_order_value|user_rank|
+-------+-------------------+-----------------+--------------------+---------+
|U002   |8750000.0          |2                |4375000.0           |1        |
|U001   |4100000.0          |3                |1366666.6666666667  |2        |
|U010   |3700000.0          |1                |3700000.0           |3        |
|U005   |1200000.0          |1                |1200000.0           |4        |
|U012   |900000.0           |1                |900000.0            |5        |
|U008   |700000.0           |1                |700000.0            |6        |
|U011   |600000.0           |1                |600000.0            |7        |
|U009   |450000.0           |1                |450000.0            |8        |
|U004   |300000.

In [30]:
#================= Perintah 1: Buat analisis cumulative GMV harian: ======================

gmv_daily_df = (
    orders_dedup
    .groupBy("order_date")
    .agg(F.sum("total_amount").alias("gmv_daily"))
    .orderBy("order_date")
)


w_cum = Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
cumulative_gmv_df = gmv_daily_df.withColumn("cumulative_gmv", F.sum("gmv_daily").over(w_cum))

print("\n=== cumulative_gmv_df dengan kolom turunan cumulative_gmv ===")
cumulative_gmv_df.show(30, truncate=False)


=== cumulative_gmv_df dengan kolom turunan cumulative_gmv ===
+----------+---------+--------------+
|order_date|gmv_daily|cumulative_gmv|
+----------+---------+--------------+
|2023-10-01|3500000.0|3500000.0     |
|2023-10-02|8500000.0|1.2E7         |
|2023-10-03|150000.0 |1.215E7       |
|2023-10-04|300000.0 |1.245E7       |
|2023-10-05|1200000.0|1.365E7       |
|2023-10-06|-50000.0 |1.36E7        |
|2023-10-07|0.0      |1.36E7        |
|2023-10-08|700000.0 |1.43E7        |
|2023-10-09|450000.0 |1.475E7       |
|2023-10-10|3700000.0|1.845E7       |
|2023-10-15|400000.0 |1.885E7       |
|2023-10-20|200000.0 |1.905E7       |
|2023-10-21|250000.0 |1.93E7        |
|2023-10-22|600000.0 |1.99E7        |
|2023-10-23|900000.0 |2.08E7        |
+----------+---------+--------------+



In [ ]:
# ========== Top 10 customers (rank 1–10) ==================
top10_customers = (
    user_summary
    .join(users_dedup, on="user_id", how="left")
    .filter(F.col("user_rank").between(1, 10))
    .select("user_id", "name", "city", "user_total_orders", "user_total_spending", "user_avg_order_value")
    .orderBy("user_total_spending", ascending=False)
)

print("\n=== top 10 customers (rank 1–10) ===")
top10_customers.show(10, truncate=False)


=== top 10 customers (rank 1–10) ===
+-------+-------------+----------+-----------------+-------------------+--------------------+
|user_id|name         |city      |user_total_orders|user_total_spending|user_avg_order_value|
+-------+-------------+----------+-----------------+-------------------+--------------------+
|U002   |Budi Santoso |surabaya  |2                |8750000.0          |4375000.0           |
|U001   |Andi Pratama |jakarta   |3                |4100000.0          |1366666.6666666667  |
|U010   |Joko Anwar   |bandung   |1                |3700000.0          |3700000.0           |
|U005   |Eka Putri    |medan     |1                |1200000.0          |1200000.0           |
|U012   |Lukman Hakim |makassar  |1                |900000.0           |900000.0            |
|U008   |Hadi Wijaya  |surabaya  |1                |700000.0           |700000.0            |
|U011   |Kiki Amelia  |yogyakarta|1                |600000.0           |600000.0            |
|U009   |Indah Sari   